# Datamine STATS Process Example

This notebook demonstrates how to configure and run the **`stats`** process wrapper in `dmstudio`.

## Process Description

## STATS

Calculate general summary parametric statistics on numeric fields in a file.

Individual fields for statistics may be selected using either the ***F1, *F2** , etc fields or may be specified in the &**FIELDLST** file. If no fields are selected then statistics will be calculated for all fields.

Ten optional keyfields are provided. If no keyfields are specified then a single set of statistics will be calculated for all data. If keyfields are specified and parameter **KEYSORT** =1 or the input file is already sorted by keyfield then statistics will be calculated for each unique combination of key values. If keyfields are specified and parameter **KEYSORT** =0 and the input file is not sorted by keyfield then data will be read from the input file until the value of one of the keyfields changes and the statistics will then be calculated for that data subset.

* **Note** (A limit of 256 fields is imposed. If more than 256 fields exist in &IN, the process will not complete.):

An optional weighting field (* **WEIGHT**) is available to weight the sample data. For example in a desurveyed drillhole file the **LENGTH** field could be used as the weighting field to give length weighted grades.

* **Note** (When calculating **MAD** and percentile statistics, **WEIGHT** is ignored.):

The variance and other moments are calculated using the large sample method (for the variance a divisor of N is used, where N is the number of samples).

The following statistics are calculated for each numeric variable :-

* total number of records in the file that meet retrieval criteria, if specified
* number of samples (excluding absent data).
* number of absent data values.
* minimum, maximum, range and mid-range.
* total and mean.
* variance, standard deviation and standard error.
* skewness and kurtosis.
* geometric mean.
* Sum, mean and variance of natural logs.
* log estimate of mean.
* coefficient of variation in percent.
* number of records equal to zero.
* number of negative value.

These results are displayed in the Command window and can also be saved to an optional &OUT file..

If the **PCNTILES** parameter is set to 1 then the 5, 10, 20, 25, 30, 40, 50, 60, 75, 80, 90 and 95 percentiles and the Median Absolute Deviation are also calculated. Processing will take longer. If this option is selected it is helpful to specify only the fields for which you wish to calculate the statistics. The Weight field is not used when calculating percentiles.

By default, statistics are calculated for all numeric variables. For example; in a typical drillhole data file containing sample co-ordinates, statistics will be calculated for both the values and the co-ordinates. The first bin in the histogram plot contains all values up to MINIMUM. The last bin contains all values above the top value. The log statistics are based on all sample values greater than, but not equal to, the system trace value.

Values of skewness and kurtosis calculated are interpreted as:

SKEWNESS |  = 0. No distortion (Gaussian).
---|---
|  > 0\. Positive skew (to the right).
|  < 0\. Negative skew (to the left).
KURTOSIS |  = 0. Mesokurtic (Gaussian).
|  > 0\. Leptokurtic (peaked).
|  < 0\. Platikurtic (flat).

### Missing Values

When the **STATS** process is run with retrieval criteria, data that is excluded by those criteria will not be reported. This is a change from previous versions which classified excluded data as "Missing Values".

The following fields are also output:

* **NSAMPLES** The number of samples in the chosen numeric (F1 etc) fields that are non-absent and used to calculate statistics.
* **NMISVALS** The number of missing records in the chosen numeric (F1 etc) fields. Samples are classified as missing if they have an absent value.
* **NUMTRACE** The number of samples in the chosen numeric (F1 etc) fields that are equal to the TRACE value
* **EQUAL0** The number of records in the chosen numeric (F1 etc) fields that contain values that = 0.
* **NEGATIVE** The number of records in the chosen numeric (F1 etc) fields that contain negative values.

### Input Files:

* **in** (Table):
  Input file.
  Required=Yes

* **fieldlst** (Undefined):
  File to supply selected fields.
  Required=No

### Output Files:

* **out** (Table):
  Output file. This will contain the fields:
  Required=No

### Fields:

* **fields** (Undefined : Undefined):
  Fields for statistics. If no fields are specified then all fields will be used.
  Default=Undefined
  Required=No

* **fieldnam** (Character : FIELDLST):
  Field in FIELDLST to identify selected fields.
  Default=Undefined
  Required=No

* **keys** (Undefined : Undefined):
  Key fields 1 for statistics.
  Default=Undefined
  Required=No

* **weight** (Numeric : IN):
  Weighting field.
  Default=Undefined
  Required=No

### Parameters:

* **keysort**:
  Set to 1 to automatically sort the data by key field. Only relevant if any key fields
  have been defined. =0 : Do not automatically sort by key fields. Use the record order of
  the input file to determine changes in key field values. =1 : Automatically sort the
  input data by key fields.
  Range=Numeric
  Values=Undefined
  Default=No
  Required=No

* **keytol**:
  The tolerance used to test whether numeric key fields are equal. All key values are
  rounded to an integer multiple of this value. If set to zero then rounding will not be
  used.
  Range=0,+
  Values=Undefined
  Default=0.00001
  Required=No

* **pcntiles**:
  Set to 1 to calculate percentiles. When calculating percentiles the process will take
  longer to run. If this option is selected it is useful to specify only the fields for
  which you wish to calculate the statistics. If this option is selected the Median
  Absolute Deviation (MAD) value is also calculated. =0 : Do not calculate percentiles. Do
  not calculate the Median Absolute Deviation. =1 : Calculate the 5, 10, 20, 25, 30, 40,
  50, 60, 75, 80, 90 and 95 percentiles and the Median Absolute Deviation.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **sortout**:
  Set to 1 to sort the output file by **FIELD** when key fields are being used. Sorting
  by **FIELD** makes it easier to compare values of variables across key fields when
  viewing the output file in the table editor. =0 : Do not sort the output file. =1 : Sort
  the output file by **FIELD**.
  Range=0,1
  Values=0,1
  Default=1
  Required=No

* **print**:
  Print flag. Default (2). 0: minimum output. 1: minimum output plus key field progress
  list. 2: full output including stats for each key field group.
  Range=0-2
  Values=0,1,2
  Default=2
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('stats')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute stats
print("Running stats...")
dm_cmd.stats(
    in_i='t_assays',  # required
    # fieldlst_i='optional',  # optional
    # out_o='t_stats_out',  # optional
    # fields_f=['AU'],  # optional
    # fieldnam_f='optional',  # optional
    # keys_f=['BHID'],  # optional
    # weight_f='optional',  # optional
    # keysort_p='No',  # optional
    # keytol_p=1e-05,  # optional
    # pcntiles_p=0,  # optional
    # sortout_p=1,  # optional
    # print_p=2,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("stats execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_stats_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")